# N052 · 表达式解析与计算

**目标：** 按优先级和结合性正确处理运算与括号。

**先修：** N023, N051。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 操作数/运算符栈；后缀表达式；一元负号；向零截断除法。

## 从问题到算法

表达式解析应先定义语法：expression处理加减，term处理乘除，factor处理一元正负、整数和括号。这样优先级由调用结构保证，不依赖碰巧的扫描顺序。逆波兰表达式则把子表达式结果压栈。整数除法向零截断，不能直接用Python的负数整除，也不要转float以免大整数精度损失。

## 最小实现

**本章接口：** `eval_rpn(tokens)`、`calculate_basic(s)`、`calculate_precedence(s)`

In [1]:
def trunc_div(a,b):
    if b==0: raise ZeroDivisionError('division by zero')
    value=abs(a)//abs(b)
    return -value if (a<0) != (b<0) else value

def eval_rpn(tokens):
    stack=[]
    for token in tokens:
        if token not in ('+','-','*','/'):
            stack.append(int(token)); continue
        b=stack.pop(); a=stack.pop()
        stack.append(a+b if token=='+' else a-b if token=='-' else a*b if token=='*' else trunc_div(a,b))
    if len(stack)!=1: raise ValueError('malformed expression')
    return stack[0]

class _Parser:
    def __init__(self,s,allow_mul): self.s=s; self.i=0; self.allow_mul=allow_mul
    def peek(self):
        while self.i<len(self.s) and self.s[self.i].isspace(): self.i+=1
        return self.s[self.i] if self.i<len(self.s) else None
    def factor(self):
        c=self.peek()
        if c in ('+','-'):
            self.i+=1; value=self.factor(); return -value if c=='-' else value
        if c=='(':
            self.i+=1; value=self.expression()
            if self.peek()!=')': raise ValueError('missing closing parenthesis')
            self.i+=1; return value
        if c is None or not '0'<=c<='9': raise ValueError('expected integer')
        value=0
        while self.i<len(self.s) and '0'<=self.s[self.i]<='9':
            value=value*10+int(self.s[self.i]); self.i+=1
        return value
    def term(self):
        value=self.factor()
        while self.allow_mul and self.peek() in ('*','/'):
            op=self.peek(); self.i+=1; other=self.factor()
            value=value*other if op=='*' else trunc_div(value,other)
        return value
    def expression(self):
        value=self.term()
        while self.peek() in ('+','-'):
            op=self.peek(); self.i+=1; other=self.term()
            value=value+other if op=='+' else value-other
        return value
    def parse(self):
        value=self.expression()
        if self.peek() is not None: raise ValueError('unconsumed input')
        return value

def calculate_basic(s): return _Parser(s,False).parse()
def calculate_precedence(s): return _Parser(s,True).parse()

## 正确性、前提与复杂度

每个解析函数消费且仅消费其语法层级的一个合法表达式；factor保证括号成为不可分的操作数，term先完成乘除，expression再合并加减。

**代价：** O(n)字符扫描；值大小固定时O(n)时间，嵌套或一元链最坏O(n)递归空间。深度极大时应改为显式栈。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tokens=['2','1','+','3','*']; stack=[]; rows=[]
for t in tokens:
    if t not in ('+','*'): stack.append(int(t))
    else:
        b=stack.pop(); a=stack.pop(); stack.append(a+b if t=='+' else a*b)
    rows.append((t,stack[:]))
show_table(['token','值栈'],rows)

token,值栈
2,[2]
1,"[2, 1]"
+,[3]
3,"[3, 3]"
*,[9]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert eval_rpn(['2','1','+','3','*'])==9
assert calculate_precedence('3+2*2')==7
assert calculate_precedence('-3/2')==-1
assert calculate_basic('1-(2-(3+4))')==6
assert calculate_precedence('(12+8)/3*2')==12
assert trunc_div(-(10**30+1),3)==-((10**30+1)//3)
for a in range(-4,5):
    for b in range(-4,5):
        assert calculate_basic(f'({a})-({b})')==a-b
        assert calculate_precedence(f'({a})*({b})+3')==a*b+3
print('N052: 所有本章断言通过')

N052: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 为一元负号构造案例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较Python整除与题意截断。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 150. Evaluate Reverse Polish Notation（canonical）
- 224. Basic Calculator（canonical）
- 227. Basic Calculator II（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。